#  Pipeline



##  Importing libraries

In [ ]:
import os, sys, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ── RDKit 
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, QED
from rdkit import RDConfig
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.rdMolDescriptors import (
    CalcTPSA, CalcNumHBD, CalcNumHBA,
    CalcNumRotatableBonds, CalcNumAromaticRings,
    CalcFractionCSP3, CalcNumRings,
    CalcNumSpiroAtoms, CalcNumAtomStereoCenters,
)

# ── ML 
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold

# ── LightGBM 
try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
    print('LightGBM available ✓')
except ImportError:
    HAS_LGB = False
    print('LightGBM not found — ensemble will use XGB + RF only')

# ── CatBoost (special ensemble for NR-AR / NR-ER) 
try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
    print('CatBoost available ✓')
except ImportError:
    HAS_CAT = False
    print('CatBoost not found — install with: pip install catboost')

# ── BalancedRandomForest (imbalanced-learn) 
try:
    from imblearn.ensemble import BalancedRandomForestClassifier
    HAS_BRF = True
    print('BalancedRandomForest available ✓')
except ImportError:
    HAS_BRF = False
    print('imbalanced-learn not found — install with: pip install imbalanced-learn')


SPECIAL_ENDPOINTS  = {'NR-AR', 'NR-AR-LBD', 'NR-ER'}
RF_STRONG_ENDPOINTS = {'NR-PPAR-gamma', 'SR-ATAD5', 'SR-HSE', 'NR-ER-LBD'}
RF_WEAK_ENDPOINTS = {'NR-AhR', 'NR-Aromatase', 'SR-ARE', 'SR-MMP', 'SR-p53'}

print(f'Special  endpoints (CAT+BRF)  : {SPECIAL_ENDPOINTS}')
print(f'RF-strong endpoints (RF=0.5)  : {RF_STRONG_ENDPOINTS}')
print(f'RF-weak   endpoints (XGB=0.6) : {RF_WEAK_ENDPOINTS}')

for d in ['eda_plots', 'models', 'results', 'plots']:
    os.makedirs(d, exist_ok=True)

plt.rcParams['figure.dpi']        = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

COLORS = ['#378ADD','#639922','#D85A30','#534AB7',
          '#0F6E56','#BA7517','#A32D2D','#185FA5',
          '#7F77DD','#1D9E75','#E24B4A','#EF9F27']

print('All imports OK')


LightGBM available ✓
CatBoost available ✓
BalancedRandomForest available ✓
Special  endpoints (CAT+BRF)  : {'NR-AR', 'NR-ER', 'NR-AR-LBD'}
RF-strong endpoints (RF=0.5)  : {'NR-PPAR-gamma', 'SR-HSE', 'SR-ATAD5', 'NR-ER-LBD'}
RF-weak   endpoints (XGB=0.6) : {'NR-Aromatase', 'SR-ARE', 'SR-p53', 'NR-AhR', 'SR-MMP'}
All imports OK


## Tox21 endpoint names

In [2]:
TOX21_ENDPOINTS = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]
print('Endpoints defined:', len(TOX21_ENDPOINTS))


Endpoints defined: 12


##  Loading data 

In [3]:
df = pd.read_csv('tox21.csv')

smiles_col = next(
    (c for c in df.columns
     if c.lower() in ['smiles', 'mol', 'structure']),
    None
)
if smiles_col is None:
    raise ValueError('No SMILES column found in tox21.csv')
if smiles_col != 'smiles':
    df = df.rename(columns={smiles_col: 'smiles'})

toxicity_labels = [e for e in TOX21_ENDPOINTS if e in df.columns]

print(f'Rows             : {len(df):,}')
print(f'Columns          : {df.shape[1]}')
print(f'Toxicity labels  : {len(toxicity_labels)}/12')
print(f'Labels found     : {toxicity_labels}')


Rows             : 7,831
Columns          : 14
Toxicity labels  : 12/12
Labels found     : ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


## Parsing SMILES

In [4]:
mols         = []
valid_rows   = []
invalid_rows = []

for i, smi in enumerate(df['smiles']):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is not None:
        mols.append(mol)
        valid_rows.append(i)
    else:
        invalid_rows.append(i)

df = df.iloc[valid_rows].reset_index(drop=True)

print(f'Valid molecules  : {len(mols):,}')
print(f'Dropped          : {len(invalid_rows)} (invalid SMILES)')
print(f'Working dataset  : {len(df):,} compounds')


[21:31:38] WARNING: not removing hydrogen atom without neighbors


Valid molecules  : 7,831
Dropped          : 0 (invalid SMILES)
Working dataset  : 7,831 compounds



## EDA — Class Balance (required for scale_pos_weight)

In [5]:
from rdkit.Chem import Descriptors

balance_stats = []
for ep in toxicity_labels:
    col     = df[ep].dropna()
    n_total = len(col)
    n_toxic = int(col.sum())
    n_safe  = n_total - n_toxic
    pct     = 100 * n_toxic / n_total if n_total > 0 else 0
    ratio   = round(n_safe / n_toxic, 1) if n_toxic > 0 else 9999
    balance_stats.append({
        'endpoint': ep, 'n_total': n_total,
        'n_toxic': n_toxic, 'n_safe': n_safe,
        'pct_toxic': round(pct, 1),
        'scale_pos_weight': ratio
    })

bal_df  = pd.DataFrame(balance_stats)
spw_dict = dict(zip(bal_df['endpoint'], bal_df['scale_pos_weight']))

print(f'{"Endpoint":<22} {"Labelled":>9} {"Toxic":>7} {"Safe":>7} {"% Toxic":>8} {"SPW":>6}')
print('-' * 62)
for _, r in bal_df.iterrows():
    print(f'{r["endpoint"]:<22} {r["n_total"]:>9,} {r["n_toxic"]:>7,} '
          f'{r["n_safe"]:>7,} {r["pct_toxic"]:>7.1f}% {r["scale_pos_weight"]:>5.1f}x')

with open('scale_pos_weights.json', 'w') as f:
    json.dump(spw_dict, f, indent=2)
print('\nSaved → scale_pos_weights.json')


Endpoint                Labelled   Toxic    Safe  % Toxic    SPW
--------------------------------------------------------------
NR-AR                      7,265     309   6,956     4.3%  22.5x
NR-AR-LBD                  6,758     237   6,521     3.5%  27.5x
NR-AhR                     6,549     768   5,781    11.7%   7.5x
NR-Aromatase               5,821     300   5,521     5.2%  18.4x
NR-ER                      6,193     793   5,400    12.8%   6.8x
NR-ER-LBD                  6,955     350   6,605     5.0%  18.9x
NR-PPAR-gamma              6,450     186   6,264     2.9%  33.7x
SR-ARE                     5,832     942   4,890    16.2%   5.2x
SR-ATAD5                   7,072     264   6,808     3.7%  25.8x
SR-HSE                     6,467     372   6,095     5.8%  16.4x
SR-MMP                     5,810     918   4,892    15.8%   5.3x
SR-p53                     6,774     423   6,351     6.2%  15.0x

Saved → scale_pos_weights.json


---
# Feature Engineering

## 1st Layer Morgan ECFP4 with chirality (radius=2, 2048 bits)



In [6]:
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, includeChirality=True
)
morgan_df = pd.DataFrame(
    [list(morgan_gen.GetFingerprint(mol)) for mol in mols],
    columns=[f'morgan_{i}' for i in range(2048)]
)
print(f'Layer 1A Morgan ECFP4 (chirality) : {morgan_df.shape}')
print(f'Avg bits ON : {morgan_df.sum(axis=1).mean():.1f} / 2048')
print(f'Sparsity    : {100*(1 - morgan_df.mean().mean()):.1f}% zeros')


Layer 1A Morgan ECFP4 (chirality) : (7831, 2048)
Avg bits ON : 30.0 / 2048
Sparsity    : 98.5% zeros


## 2nd Layer — Atom Pair Fingerprints (1024 bits)




In [7]:
from rdkit.Chem import rdMolDescriptors

def get_atom_pair_fp(mol):
    fp = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(
        mol, nBits=1024, includeChirality=True
    )
    return list(fp)

ap_df = pd.DataFrame(
    [get_atom_pair_fp(mol) for mol in mols],
    columns=[f'ap_{i}' for i in range(1024)]
)
print(f'Layer 1B Atom Pair fingerprints : {ap_df.shape}')
print(f'Avg bits ON : {ap_df.sum(axis=1).mean():.1f} / 1024')


Layer 1B Atom Pair fingerprints : (7831, 1024)
Avg bits ON : 132.7 / 1024


## 3rd Layer — Topological Torsion Fingerprints (1024 bits)




In [8]:
def get_torsion_fp(mol):
    fp = rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(
        mol, nBits=1024, includeChirality=True
    )
    return list(fp)

tt_df = pd.DataFrame(
    [get_torsion_fp(mol) for mol in mols],
    columns=[f'tt_{i}' for i in range(1024)]
)
print(f'Layer 1C Topological Torsion : {tt_df.shape}')
print(f'Avg bits ON : {tt_df.sum(axis=1).mean():.1f} / 1024')


Layer 1C Topological Torsion : (7831, 1024)
Avg bits ON : 28.0 / 1024


## 4th Layer — MACCS Structural Keys (167 bits)

In [9]:
maccs_df = pd.DataFrame(
    [list(MACCSkeys.GenMACCSKeys(mol)) for mol in mols],
    columns=[f'maccs_{i}' for i in range(167)]
)
print(f'Shape     : {maccs_df.shape}')
print(f'NaN count : {maccs_df.isna().sum().sum()} (always 0)')


Shape     : (7831, 167)
NaN count : 0 (always 0)


## 5th Layer — Core RDKit Descriptors (10 properties)

The original 10 physicochemical descriptors: MW, logP, QED, TPSA, HBD, HBA,
RotBonds, ArRings, FracCSP3, HeavyAtoms.


In [10]:
DESC_COLUMNS = [
    'MW','logP','QED','TPSA','HBD','HBA',
    'RotBonds','ArRings','FracCSP3','HeavyAtoms'
]

def compute_descriptors(mol):
    try:
        return {
            'MW'        : Descriptors.MolWt(mol),
            'logP'      : Descriptors.MolLogP(mol),
            'QED'       : QED.qed(mol),
            'TPSA'      : CalcTPSA(mol),
            'HBD'       : CalcNumHBD(mol),
            'HBA'       : CalcNumHBA(mol),
            'RotBonds'  : CalcNumRotatableBonds(mol),
            'ArRings'   : CalcNumAromaticRings(mol),
            'FracCSP3'  : CalcFractionCSP3(mol),
            'HeavyAtoms': mol.GetNumHeavyAtoms(),
        }
    except Exception:
        return {col: np.nan for col in DESC_COLUMNS}

desc_df = pd.DataFrame([compute_descriptors(mol) for mol in mols])
print(f'Shape     : {desc_df.shape}')
print(f'NaN count : {desc_df.isna().sum().sum()}')
print(desc_df[['MW','logP','QED','TPSA']].describe().round(2))


[21:32:14] WARNING: not removing hydrogen atom without neighbors


Shape     : (7831, 10)
NaN count : 0
            MW     logP      QED     TPSA
count  7831.00  7831.00  7831.00  7831.00
mean    276.32     2.37     0.55    59.62
std     165.82     2.37     0.19    58.95
min       9.01   -44.16     0.01     0.00
25%     165.21     1.15     0.42    26.30
50%     240.30     2.37     0.55    46.53
75%     343.04     3.65     0.69    77.08
max    1999.07    22.61     0.94  1095.85


## 6th Layer — Extended RDKit Descriptors (~200 properties) 



RDKit has ~200 2D descriptors covering:
- **VSA descriptors** (MOE-style surface area bins by logP/charge/H-bond)
- **Estate indices** (electrochemical atom environments)
- **Ring system descriptors** (bridgehead atoms, spiro atoms)
- **Connectivity indices** (chi0, chi1... molecular graph topology)
- **Kappa shape indices** (branching and flexibility)
- **Fragment counts** (number of each functional group type)



In [ ]:

_all_desc = Descriptors.descList  

_exclude = set(DESC_COLUMNS) | {'MolLogP', 'MolWt'}

def compute_extended_descriptors(mol):
    result = {}
    for name, func in _all_desc:
        if name in _exclude:
            continue
        try:
            val = func(mol)
            
            if val is not None and np.isfinite(float(val)):
                result[name] = float(val)
            else:
                result[name] = np.nan
        except Exception:
            result[name] = np.nan
    return result

print('Computing extended descriptors for all molecules...')
ext_rows = [compute_extended_descriptors(mol) for mol in mols]
ext_df   = pd.DataFrame(ext_rows)
ext_df   = ext_df.replace([np.inf, -np.inf], np.nan)

nan_frac  = ext_df.isna().mean()
keep_cols = nan_frac[nan_frac < 0.5].index
ext_df    = ext_df[keep_cols]

print(f'Shape after NaN filter  : {ext_df.shape}')
print(f'NaN count remaining     : {ext_df.isna().sum().sum()}')
print(f'Descriptors included    : {list(ext_df.columns[:10])} ...')


Computing extended descriptors for all molecules...


[21:32:29] WARNING: not removing hydrogen atom without neighbors
[21:32:29] WARNING: not removing hydrogen atom without neighbors


Shape after NaN filter  : (7831, 205)
NaN count remaining     : 1634
Descriptors included    : ['MaxEStateIndex', 'MinEStateIndex', 'MaxAbsEStateIndex', 'MinAbsEStateIndex', 'qed', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons', 'MaxPartialCharge'] ...


## 7th Layer — NR-ER SMARTS Fragment Counts 

These 8 SMARTS patterns encode the pharmacophore features that define
estrogen receptor binding — phenols, hydroxyl groups, steroid-like rings.

All 2D. No conformer. Direct biological signal for NR-ER and NR-ER-LBD.

|


In [12]:
from rdkit.Chem import MolFromSmarts

# SMARTS patterns — biologically motivated for NR-ER / NR-ER-LBD
NR_ER_SMARTS = {
    'smarts_phenol'          : MolFromSmarts('c1ccc(O)cc1'),          # phenol
    'smarts_aliphatic_OH'    : MolFromSmarts('[C;!a][OH]'),            # aliphatic hydroxyl
    'smarts_aromatic_ring'   : MolFromSmarts('c1ccccc1'),              # benzene ring
    'smarts_fused_6_6'       : MolFromSmarts('c1ccc2ccccc2c1'),        # fused 6-6 aromatic
    'smarts_steroid_ABC'     : MolFromSmarts('[C]1CC[C]2CC[C]3CC[C@@H](CC3)[C@@H]2[C@@H]1'), # steroid proxy
    'smarts_halogenated_aryl': MolFromSmarts('c1ccc([F,Cl,Br,I])cc1'), # halogenated phenyl
    'smarts_amine'           : MolFromSmarts('[NX3;H2,H1;!$(NC=O)]'),  # primary/secondary amine
    'smarts_carbonyl'        : MolFromSmarts('[C;!R]=O'),              # non-ring carbonyl
}

def compute_smarts_features(mol):
    result = {}
    for name, patt in NR_ER_SMARTS.items():
        try:
            if patt is None:
                result[name] = 0
            else:
                matches = mol.GetSubstructMatches(patt)
                result[name] = len(matches)
        except Exception:
            result[name] = 0
    return result

smarts_df = pd.DataFrame([compute_smarts_features(mol) for mol in mols])
print(f'Layer 3C SMARTS fragment counts : {smarts_df.shape}')
print(smarts_df.describe().round(2))


Layer 3C SMARTS fragment counts : (7831, 8)
       smarts_phenol  smarts_aliphatic_OH  smarts_aromatic_ring  \
count        7831.00              7831.00               7831.00   
mean            0.36                 0.49                  0.84   
std             0.90                 1.23                  0.93   
min             0.00                 0.00                  0.00   
25%             0.00                 0.00                  0.00   
50%             0.00                 0.00                  1.00   
75%             0.00                 1.00                  1.00   
max            30.00                24.00                 10.00   

       smarts_fused_6_6  smarts_steroid_ABC  smarts_halogenated_aryl  \
count           7831.00              7831.0                  7831.00   
mean               0.04                 0.0                     0.23   
std                0.29                 0.0                     0.71   
min                0.00                 0.0                     

## 8th Layer — Engineered Features 

In [13]:
eng = pd.DataFrame()

eng['lip_pass']       = (
    (desc_df['MW'] < 500) & (desc_df['logP'] < 5) &
    (desc_df['HBD'] <= 5) & (desc_df['HBA'] <= 10)
).astype(int)

eng['lip_violations'] = (
    (desc_df['MW'] >= 500).astype(int) +
    (desc_df['logP'] >= 5).astype(int) +
    (desc_df['HBD'] > 5).astype(int)  +
    (desc_df['HBA'] > 10).astype(int)
)

eng['logP_x_MW']   = desc_df['logP'] * desc_df['MW']
eng['TPSA_per_MW'] = desc_df['TPSA'] / (desc_df['MW'] + 1)
eng['HBD_HBA_sum'] = desc_df['HBD'] + desc_df['HBA']
eng['logMW']       = np.log1p(desc_df['MW'])

eng = eng.replace([np.inf, -np.inf], np.nan)

print(f'Shape     : {eng.shape}')
print(f'Features  : {list(eng.columns)}')
print(f'NaN count : {eng.isna().sum().sum()}')


Shape     : (7831, 6)
Features  : ['lip_pass', 'lip_violations', 'logP_x_MW', 'TPSA_per_MW', 'HBD_HBA_sum', 'logMW']
NaN count : 0


## 9th Layer — ZINC-style Drug-likeness Properties

In [14]:
try:
    from rdkit import RDConfig
    sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
    import sascorer
    HAS_SAS = True
    print('SAS scorer loaded ✓')
except Exception:
    HAS_SAS = False
    print('SAS scorer not found — using structural estimate')

def compute_zinc_features(mol):
    try:
        mw   = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        qed  = QED.qed(mol)
        if HAS_SAS:
            try:    sas = sascorer.calculateScore(mol)
            except: sas = np.nan
        else:
            sas = min(max(
                1.0 + 0.25*CalcNumRings(mol)
                    + 0.04*mol.GetNumHeavyAtoms()
                    + 0.5*CalcNumAtomStereoCenters(mol)
                    + 0.8*CalcNumSpiroAtoms(mol),
                1.0), 10.0)
        return {
            'zinc_SAS'     : sas,
            'drug_like'    : int(150<mw<500 and -0.4<logp<5.6),
            'fragment_like': int(mw<300 and logp<3),
            'lead_like'    : int(250<mw<350 and -1<logp<3.5),
        }
    except Exception:
        return {k: np.nan for k in
                ['zinc_SAS','drug_like','fragment_like','lead_like']}

zinc_df = pd.DataFrame([compute_zinc_features(mol) for mol in mols])
zinc_df = zinc_df.replace([np.inf, -np.inf], np.nan)

print(f'Shape     : {zinc_df.shape}')
print(f'NaN count : {zinc_df.isna().sum().sum()}')
n_drug  = zinc_df['drug_like'].sum()
print(f'Drug-like : {int(n_drug):,}/{len(zinc_df):,} ({100*n_drug/len(zinc_df):.1f}%)')


SAS scorer loaded ✓


[21:33:35] WARNING: not removing hydrogen atom without neighbors


Shape     : (7831, 4)
NaN count : 0
Drug-like : 5,143/7,831 (65.7%)


## 10th Layer — Chirality + Formal Charge




In [15]:
def compute_chirality_charge(mol):
    try:
        atoms   = list(mol.GetAtoms())
        charges = [a.GetFormalCharge() for a in atoms]
        n_cw    = sum(1 for a in atoms if int(a.GetChiralTag()) == 1)
        n_ccw   = sum(1 for a in atoms if int(a.GetChiralTag()) == 2)
        return {
            'chiral_n_R'      : n_cw,
            'chiral_n_S'      : n_ccw,
            'chiral_total'    : n_cw + n_ccw,
            'has_chirality'   : int((n_cw + n_ccw) > 0),
            'chiral_RS_ratio' : n_cw / (n_ccw + 1e-6),
            'total_charge'    : sum(charges),
            'n_pos_atoms'     : sum(1 for c in charges if c > 0),
            'n_neg_atoms'     : sum(1 for c in charges if c < 0),
            'max_atom_charge' : max(charges) if charges else 0,
            'charge_range'    : max(charges) - min(charges) if charges else 0,
        }
    except Exception:
        return {k: np.nan for k in [
            'chiral_n_R','chiral_n_S','chiral_total','has_chirality',
            'chiral_RS_ratio','total_charge','n_pos_atoms','n_neg_atoms',
            'max_atom_charge','charge_range'
        ]}

chiral_df = pd.DataFrame([compute_chirality_charge(mol) for mol in mols])
print(f'Shape     : {chiral_df.shape}')
print(f'NaN count : {chiral_df.isna().sum().sum()}')
print(f'Molecules with chirality : {int(chiral_df["has_chirality"].sum()):,} '
      f'({100*chiral_df["has_chirality"].mean():.1f}%)')


Shape     : (7831, 10)
NaN count : 0
Molecules with chirality : 1,322 (16.9%)


---
## Combining All Layers + KNN Imputation + VarianceThreshold



## 11th Layer -  AUTOCORR3D + WHIM 3D Descriptors


In [ ]:
mols_3d = []
for i, mol in enumerate(mols):
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    try:
        AllChem.MMFFOptimizeMolecule(mol)
        mols_3d.append(mol)
    except Exception:
        mols_3d.append(None) 

print(f'Generated 3D conformers for {len(mols_3d)} molecules. ({mols_3d.count(None)} failed)')

[21:33:49] UFFTYPER: Unrecognized charge state for atom: 0
[21:33:49] UFFTYPER: Unrecognized atom type: Zn+2 (0)
[21:33:51] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[21:33:52] UFFTYPER: Unrecognized charge state for atom: 6
[21:33:52] WARNING: not removing hydrogen atom without neighbors
[21:33:56] UFFTYPER: Unrecognized charge state for atom: 7
[21:33:56] UFFTYPER: Unrecognized atom type: Cu6+1 (0)
[21:34:00] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[21:34:00] UFFTYPER: Unrecognized charge state for atom: 0
[21:34:00] UFFTYPER: Unrecognized charge state for atom: 4
[21:34:01] UFFTYPER: Unrecognized charge state for atom: 4
[21:34:25] UFFTYPER: Unrecognized charge state for atom: 8
[21:34:27] UFFTYPER: Warning: hybridization set to SP for atom 0
[21:34:27] UFFTYPER: Unrecognized charge state for atom: 0
[21:34:29] UFFTYPER: Unrecognized atom type: Cr3+3 (1)
[21:34:29] UFFTYPER: Unrecognized atom type: Cr3+3 (5)
[21:34:33] UFFTYPER: Unrecognized atom type: Cr3+3 (4)
[21:

Generated 3D conformers for 7831 molecules. (21 failed)


In [ ]:
from rdkit.Chem import rdMolDescriptors

def compute_autocorr3d(mol_3d):
    """
    Returns 80 autocorrelation values.
    Each value = correlation of an atomic property at a given distance.
    Handles failed conformers by returning NaN row.
    """
    if mol_3d is None:
        return {f'autocorr3d_{i}': np.nan for i in range(80)}
    try:
        ac = rdMolDescriptors.CalcAUTOCORR3D(mol_3d)
        return {f'autocorr3d_{i}': v for i, v in enumerate(ac)}
    except Exception:
        return {f'autocorr3d_{i}': np.nan for i in range(80)}

autocorr3d_df = pd.DataFrame([compute_autocorr3d(m) for m in mols_3d])
print(f'AUTOCORR3D shape : {autocorr3d_df.shape}')
print(f'NaN count        : {autocorr3d_df.isna().sum().sum()}')


def compute_whim(mol_3d):
    """
    Returns 114 WHIM descriptor values.
    Handles failed conformers by returning NaN row.
    """
    if mol_3d is None:
        return {f'whim_{i}': np.nan for i in range(114)}
    try:
        whim = rdMolDescriptors.CalcWHIM(mol_3d)
        return {f'whim_{i}': v for i, v in enumerate(whim)}
    except Exception:
        return {f'whim_{i}': np.nan for i in range(114)}

whim_df = pd.DataFrame([compute_whim(m) for m in mols_3d])
print(f'WHIM shape before VT : {whim_df.shape}')
print(f'NaN count            : {whim_df.isna().sum().sum()}')

whim_vals = whim_df.values.copy()
col_medians = np.nanmedian(whim_vals, axis=0)
nan_mask = np.isnan(whim_vals)
whim_vals[nan_mask] = np.take(col_medians, np.where(nan_mask)[1])

vt_whim = VarianceThreshold(threshold=0.01)
whim_filtered = vt_whim.fit_transform(whim_vals)
kept_whim_cols = [whim_df.columns[i] for i in range(114) if vt_whim.get_support()[i]]
whim_filtered_df = pd.DataFrame(whim_filtered, columns=kept_whim_cols)
print(f'WHIM shape after VT  : {whim_filtered_df.shape}')
print(f'Removed {114 - whim_filtered_df.shape[1]} near-zero WHIM features')

autocorr_vals = autocorr3d_df.values.copy()
col_med_ac = np.nanmedian(autocorr_vals, axis=0)
nan_mask_ac = np.isnan(autocorr_vals)
autocorr_vals[nan_mask_ac] = np.take(col_med_ac, np.where(nan_mask_ac)[1])
autocorr3d_clean_df = pd.DataFrame(autocorr_vals, columns=autocorr3d_df.columns)

print(f'\nFinal 3D feature counts:')
print(f'  AUTOCORR3D : {autocorr3d_clean_df.shape[1]} features')
print(f'  WHIM       : {whim_filtered_df.shape[1]} features (after VT)')
print(f'  Total 3D   : {autocorr3d_clean_df.shape[1] + whim_filtered_df.shape[1]} features')

[21:45:00] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 238 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Descriptors\AUTOCORR3D.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[21:45:00] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 238 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Descriptors\AUTOCORR3D.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[21:45:01] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 238 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Descriptors\AUTOCORR3D.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[21:45:01] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 238 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Descriptors\AUTOCORR3D.cpp
Failed Expression: mo

AUTOCORR3D shape : (7831, 80)
NaN count        : 3120


[21:45:02] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 382 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Descriptors\WHIM.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[21:45:02] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 382 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Descriptors\WHIM.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[21:45:02] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 382 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Descriptors\WHIM.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[21:45:02] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 382 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Descriptors\WHIM.cpp
Failed Expression: mol.getNumConformers() >= 

WHIM shape before VT : (7831, 114)
NaN count            : 3393
WHIM shape after VT  : (7831, 100)
Removed 14 near-zero WHIM features

Final 3D feature counts:
  AUTOCORR3D : 80 features
  WHIM       : 100 features (after VT)
  Total 3D   : 180 features


In [ ]:

# FINAL AUGMENTATION OF LAYERS


# --- Binary fingerprint layers (VarianceThreshold applied) ---
fp_matrix = pd.concat([
    morgan_df.reset_index(drop=True),   # Layer 1A: 2048 bits ECFP4+chiral
    ap_df.reset_index(drop=True),       # Layer 1B: 1024 bits Atom Pair  (NEW)
    tt_df.reset_index(drop=True),       # Layer 1C: 1024 bits Topo Torsion (NEW)
    maccs_df.reset_index(drop=True),    # Layer 2:   167 bits MACCS
], axis=1)

print(f'Total fingerprint bits before VarianceThreshold : {fp_matrix.shape[1]}')

# VarianceThreshold: remove bits ON in <0.1% or >99.9% of molecules
# threshold = p*(1-p) where p=0.001  →  threshold = 0.000999
vt = VarianceThreshold(threshold=0.001 * (1 - 0.001))
fp_filtered = vt.fit_transform(fp_matrix.values)

kept_fp_cols = [fp_matrix.columns[i] for i in range(fp_matrix.shape[1])
                if vt.get_support()[i]]
fp_filtered_df = pd.DataFrame(fp_filtered, columns=kept_fp_cols)

print(f'Total fingerprint bits after  VarianceThreshold : {fp_filtered_df.shape[1]}')
print(f'Removed {fp_matrix.shape[1] - fp_filtered_df.shape[1]} near-zero bits')

# --- Continuous layers (KNN imputation applied) ---
continuous_df = pd.concat([
    desc_df.reset_index(drop=True),         
    ext_df.reset_index(drop=True),          
    smarts_df.reset_index(drop=True),       
    eng.reset_index(drop=True),             
    zinc_df.reset_index(drop=True),         
    chiral_df.reset_index(drop=True),       
    autocorr3d_clean_df.reset_index(drop=True),  
    whim_filtered_df.reset_index(drop=True),     
], axis=1)

print(f'\nContinuous features : {continuous_df.shape[1]} cols')
n_nan = continuous_df.isna().sum().sum()
print(f'NaN in continuous   : {n_nan}')

if n_nan > 0:
    knn = KNNImputer(n_neighbors=5, weights='distance')
    cont_imputed = pd.DataFrame(
        knn.fit_transform(continuous_df.values),
        columns=continuous_df.columns
    )
    print(f'NaN after KNN imputation : {cont_imputed.isna().sum().sum()} (must be 0)')
else:
    cont_imputed = continuous_df.copy()
    print('No NaN — KNN skipped')


cont_imputed = cont_imputed.replace([np.inf, -np.inf], np.nan)

cont_imputed = cont_imputed.fillna(cont_imputed.median())

cont_imputed = cont_imputed.clip(lower=-1e6, upper=1e6)


feature_matrix = pd.concat([
    fp_filtered_df.reset_index(drop=True),
    cont_imputed.reset_index(drop=True),
], axis=1)

feat_cols    = feature_matrix.columns.tolist()
label_matrix = df[toxicity_labels].reset_index(drop=True)

print(f'\n{"="*55}')
print(f'FINAL FEATURE MATRIX')
print(f'{"="*55}')
print(f'  Layer 1A Morgan ECFP4+chiral  : 2048 bits (before VT)')
print(f'  Layer 1B Atom Pair            : 1024 bits (NEW, before VT)')
print(f'  Layer 1C Topological Torsion  : 1024 bits (NEW, before VT)')
print(f'  Layer 2  MACCS                :  167 bits (before VT)')
print(f'  After VarianceThreshold       : {fp_filtered_df.shape[1]} bits kept')
print(f'  Layer 3A Core descriptors     : {desc_df.shape[1]} cols')
print(f'  Layer 3B Extended descriptors : {ext_df.shape[1]} cols')
print(f'  Layer 3C SMARTS NR-ER         : {smarts_df.shape[1]} cols (NEW)')
print(f'  Layer 4  Engineered           : {eng.shape[1]} cols')
print(f'  Layer 5  ZINC                 : {zinc_df.shape[1]} cols')
print(f'  +10      Chirality/charge     : 10 cols')
print(f'  Layer 6A AUTOCORR3D           : {autocorr3d_clean_df.shape[1]} cols (NEW)')
print(f'  Layer 6B WHIM (post-VT)       : {whim_filtered_df.shape[1]} cols (NEW)')
print(f'  TOTAL                         : {len(feat_cols)} cols')
print(f'{"="*55}')

# Save
final_df = pd.concat([
    df[['smiles']].reset_index(drop=True),
    feature_matrix,
    label_matrix,
], axis=1)
final_df.to_csv('features_raw.csv', index=False)
with open('feature_cols.json', 'w') as f:
    json.dump(feat_cols, f)
print('\nSaved → features_raw.csv')
print('Saved → feature_cols.json')


Total fingerprint bits before VarianceThreshold : 4263
Total fingerprint bits after  VarianceThreshold : 3974
Removed 289 near-zero bits

Continuous features : 423 cols
NaN in continuous   : 1634
NaN after KNN imputation : 0 (must be 0)

FINAL FEATURE MATRIX
  Layer 1A Morgan ECFP4+chiral  : 2048 bits (before VT)
  Layer 1B Atom Pair            : 1024 bits (NEW, before VT)
  Layer 1C Topological Torsion  : 1024 bits (NEW, before VT)
  Layer 2  MACCS                :  167 bits (before VT)
  After VarianceThreshold       : 3974 bits kept
  Layer 3A Core descriptors     : 10 cols
  Layer 3B Extended descriptors : 205 cols
  Layer 3C SMARTS NR-ER         : 8 cols (NEW)
  Layer 4  Engineered           : 6 cols
  Layer 5  ZINC                 : 4 cols
  +10      Chirality/charge     : 10 cols
  Layer 6A AUTOCORR3D           : 80 cols (NEW)
  Layer 6B WHIM (post-VT)       : 100 cols (NEW)
  TOTAL                         : 4397 cols

Saved → features_raw.csv
Saved → feature_cols.json


---
## Testing Model 1 — XGBoost Baseline with 5-fold Cross-Validation (Baseline Evaluation)


In [19]:

# STAGE 1 — XGBoost baseline 

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
    print('SMOTE available ✓')
except ImportError:
    HAS_SMOTE = False
    print('SMOTE not found — install with: pip install imbalanced-learn')

X_array = feature_matrix.values.astype(np.float32)
np.nan_to_num(X_array, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
print(f"X_array shape : {X_array.shape}")
print(f"inf count     : {np.isinf(X_array).sum()} (must be 0)")
print(f"nan count     : {np.isnan(X_array).sum()} (must be 0)")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Cache structures (reused by ensemble cell) ────────────────────
# xgb_fold_preds[ep]  = np.array of OOF predictions (full y length)
# xgb_fold_aucs[ep]   = list of 5 fold AUC scores
# xgb_fold_models[ep] = list of 5 trained XGBClassifier objects
# xgb_final_models    = dict of final XGB trained on full data
# ep_splits[ep]       = (mask, X_ep, y_ep, spw) — reused in ensemble
xgb_fold_preds  = {}
xgb_fold_aucs   = {}
xgb_fold_models = {}
xgb_models      = {}   # final models trained on full data
ep_data         = {}   # (mask, X_ep, y_ep, spw) per endpoint

xgb_results = []
oof_probs   = {}
oof_labels  = {}

print('\nXGBOOST — 5-fold CV (results cached for ensemble)')
print(f'{"Endpoint":<22} {"N rows":>7} {"Toxic":>7} {"SPW":>6} {"AUC":>8}')
print('-' * 56)

for ep in toxicity_labels:
    mask    = label_matrix[ep].notna()
    X_ep    = X_array[mask]
    y_ep    = label_matrix[ep][mask].values.astype(int)
    n_toxic = int((y_ep == 1).sum())
    n_safe  = int((y_ep == 0).sum())

    if n_toxic < 10:
        print(f'{ep:<22} SKIPPED ({n_toxic} toxic)')
        continue

    spw = round(n_safe / n_toxic, 1)
    ep_data[ep] = (mask, X_ep, y_ep, spw)   # cache for ensemble

    fold_aucs  = []
    fold_preds = np.zeros(len(y_ep))
    fold_mods  = []

    for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_ep, y_ep)):
        X_tr_fold = X_ep[tr_idx]
        y_tr_fold = y_ep[tr_idx]
        X_val_fold = X_ep[val_idx]
        y_val_fold = y_ep[val_idx]

        
        if HAS_SMOTE:
            n_toxic_fold = (y_tr_fold == 1).sum()
            n_safe_fold  = (y_tr_fold == 0).sum()
            
            if n_toxic_fold >= 6:
                target_ratio = min(0.3, (n_toxic_fold * 2) / n_safe_fold)
                smote = SMOTE(
                    sampling_strategy = target_ratio,
                    k_neighbors       = min(5, n_toxic_fold - 1),
                    random_state      = 42
                )
                try:
                    X_tr_fold, y_tr_fold = smote.fit_resample(X_tr_fold, y_tr_fold)
                except Exception:
                    pass  

        m = XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
            scale_pos_weight=spw, eval_metric='auc',
            random_state=42, n_jobs=-1, verbosity=0,
        )
        m.fit(X_tr_fold, y_tr_fold)
        probs = m.predict_proba(X_val_fold)[:, 1]
        fold_preds[val_idx] = probs
        fold_aucs.append(roc_auc_score(y_val_fold, probs))
        fold_mods.append(m)   # cache trained fold model

    # Store cache
    xgb_fold_preds[ep]  = fold_preds
    xgb_fold_aucs[ep]   = fold_aucs
    xgb_fold_models[ep] = fold_mods

    mean_auc = float(np.mean(fold_aucs))
    oof_probs[ep]  = fold_preds
    oof_labels[ep] = y_ep

    xgb_results.append({'endpoint': ep, 'n_rows': len(y_ep),
                        'n_toxic': n_toxic, 'spw': spw,
                        'xgb_auc': round(mean_auc, 4)})
    print(f'{ep:<22} {len(y_ep):>7,} {n_toxic:>7,} '
          f'{spw:>5.1f}x {mean_auc:>8.4f}')

    
    final_xgb = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        scale_pos_weight=spw, eval_metric='auc',
        random_state=42, n_jobs=-1, verbosity=0,
    )
    final_xgb.fit(X_ep, y_ep)
    xgb_models[ep] = final_xgb

xgb_df   = pd.DataFrame(xgb_results)
mean_xgb = xgb_df['xgb_auc'].mean()
print('-' * 56)
print(f'{"XGB MEAN AUC":<45} {mean_xgb:>8.4f}')
print(f'\nCached: {len(xgb_fold_preds)} endpoints × 5 folds = '
      f'{len(xgb_fold_preds)*5} XGB fold models stored')
print('Ensemble cell will retrieve these — XGB will NOT retrain.')
with open('models/xgb_models.pkl', 'wb') as f:
    pickle.dump(xgb_models, f)
print('Saved → models/xgb_models.pkl')


SMOTE available ✓
X_array shape : (7831, 4397)
inf count     : 0 (must be 0)
nan count     : 0 (must be 0)

XGBOOST — 5-fold CV (results cached for ensemble)
Endpoint                N rows   Toxic    SPW      AUC
--------------------------------------------------------
NR-AR                    7,265     309  22.5x   0.7887
NR-AR-LBD                6,758     237  27.5x   0.8596
NR-AhR                   6,549     768   7.5x   0.9108
NR-Aromatase             5,821     300  18.4x   0.8707
NR-ER                    6,193     793   6.8x   0.7291
NR-ER-LBD                6,955     350  18.9x   0.8431
NR-PPAR-gamma            6,450     186  33.7x   0.8572
SR-ARE                   5,832     942   5.2x   0.8513
SR-ATAD5                 7,072     264  25.8x   0.8916
SR-HSE                   6,467     372  16.4x   0.8249
SR-MMP                   5,810     918   5.3x   0.9265
SR-p53                   6,774     423  15.0x   0.8830
--------------------------------------------------------
XGB MEAN AUC 


## Stage 2 — Cross-Endpoint Meta-Feature Construction


In [20]:

# STAGE 2 MODEL — Cross-Endpoint OOF Meta-Features

oof_feature_rows = {ep: np.full(len(df), np.nan) for ep in oof_probs}

for ep, probs in oof_probs.items():
    mask = label_matrix[ep].notna()
    idx  = np.where(mask)[0]
    oof_feature_rows[ep][idx] = probs

oof_feat_df = pd.DataFrame(oof_feature_rows,
                            columns=[f'oof_{ep}' for ep in oof_probs])
oof_feat_df.columns = [f'oof_{ep}' for ep in oof_probs]
oof_feat_df = oof_feat_df.fillna(oof_feat_df.mean())

print(f'Cross-endpoint meta-feature matrix : {oof_feat_df.shape}')
print(f'Columns : {list(oof_feat_df.columns)}')

X_array_aug = np.hstack([X_array, oof_feat_df.values.astype(np.float32)])
np.nan_to_num(X_array_aug, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

print(f'\nOriginal X shape  : {X_array.shape}')
print(f'Augmented X shape : {X_array_aug.shape}  '
      f'(+{X_array_aug.shape[1] - X_array.shape[1]} meta-features)')

ep_data_aug = {}
for ep, (mask, X_ep, y_ep, spw) in ep_data.items():
    ep_data_aug[ep] = (mask, X_array_aug[mask], y_ep, spw)

print('ep_data_aug ready for Stage 2.')


Cross-endpoint meta-feature matrix : (0, 12)
Columns : ['oof_NR-AR', 'oof_NR-AR-LBD', 'oof_NR-AhR', 'oof_NR-Aromatase', 'oof_NR-ER', 'oof_NR-ER-LBD', 'oof_NR-PPAR-gamma', 'oof_SR-ARE', 'oof_SR-ATAD5', 'oof_SR-HSE', 'oof_SR-MMP', 'oof_SR-p53']


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 7831 and the array at index 1 has size 0


## Stage 3 — Endpoint-Aware Ensemble



In [ ]:

ensemble_results = []
lgb_models       = {}
rf_models        = {}
cat_models       = {}
brf_models       = {}
ensemble_models  = {}

print('ENDPOINT-AWARE ENSEMBLE (XGB from cache — no retraining)')
print(f'{"Endpoint":<22} {"Tier":<10} {"XGB":>7} {"M2":>8} {"M3":>8} {"Ensemble":>10}')
print('-' * 72)

for ep in toxicity_labels:
    # ── Retrieve XGB cache ────────────────────────────────────────
    if ep not in ep_data:
        continue   # was skipped in Stage 1

    mask, X_ep, y_ep, spw = ep_data_aug.get(ep, ep_data[ep])  
    cached_xgb_preds = xgb_fold_preds[ep]   
    cached_xgb_aucs  = xgb_fold_aucs[ep]    
    final_xgb2       = xgb_models[ep]        

    is_special   = ep in SPECIAL_ENDPOINTS
    is_rf_strong = ep in RF_STRONG_ENDPOINTS

    if is_special:
        tier = 'CAT+BRF'
    elif is_rf_strong:
        tier = 'RF-heavy'
    else:
        tier = 'XGB-heavy'

    fold_m2  = []
    fold_m3  = []
    fold_ens = []

    for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_ep, y_ep)):
        X_tr, X_val = X_ep[tr_idx], X_ep[val_idx]
        y_tr, y_val = y_ep[tr_idx], y_ep[val_idx]

        
        p_xgb = cached_xgb_preds[val_idx] 

        # Model 2: CatBoost (special) OR LightGBM (standard) 
        if is_special and HAS_CAT:
            _m2 = CatBoostClassifier(
                iterations=300, depth=6, learning_rate=0.05,
                auto_class_weights='Balanced',
                eval_metric='AUC',
                random_seed=42, thread_count=-1, verbose=0,
            )
            _m2.fit(X_tr, y_tr)
            p_m2 = _m2.predict_proba(X_val)[:, 1]
        elif HAS_LGB:
            _m2 = LGBMClassifier(
                n_estimators=300, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8,
                scale_pos_weight=spw,
                random_state=42, n_jobs=-1, verbose=-1,
            )
            _m2.fit(X_tr, y_tr)
            p_m2 = _m2.predict_proba(X_val)[:, 1]
        else:
            p_m2 = p_xgb  # fallback

        #  Model 3: BalancedRF (special) OR standard RF
        if is_special and HAS_BRF:
            _m3 = BalancedRandomForestClassifier(
                n_estimators=200, max_features='sqrt',
                sampling_strategy='auto', replacement=False,
                random_state=42, n_jobs=-1,
            )
            _m3.fit(X_tr, y_tr)
            p_m3 = _m3.predict_proba(X_val)[:, 1]
        else:
            _m3 = RandomForestClassifier(
                n_estimators=200, max_features='sqrt',
                class_weight='balanced',
                random_state=42, n_jobs=-1,
            )
            _m3.fit(X_tr, y_tr)
            p_m3 = _m3.predict_proba(X_val)[:, 1]

        if is_special:
            p_ens = 0.50*p_m3 + 0.35*p_xgb + 0.15*p_m2
        elif is_rf_strong:
            p_ens = 0.45*p_m3 + 0.45*p_xgb + 0.10*p_m2
        else:
            p_ens = 0.65*p_xgb + 0.25*p_m3 + 0.10*p_m2

        fold_m2.append(roc_auc_score(y_val, p_m2))
        fold_m3.append(roc_auc_score(y_val, p_m3))
        fold_ens.append(roc_auc_score(y_val, p_ens))

    auc_xgb = float(np.mean(cached_xgb_aucs))
    auc_m2  = float(np.mean(fold_m2))
    auc_m3  = float(np.mean(fold_m3))
    auc_ens = float(np.mean(fold_ens))

    final_m2    = None
    final_m3    = None
    ens_weights = None

    if is_special:
        if HAS_CAT:
            final_m2 = CatBoostClassifier(
                iterations=300, depth=6, learning_rate=0.05,
                auto_class_weights='Balanced', eval_metric='AUC',
                random_seed=42, thread_count=-1, verbose=0,
            )
            final_m2.fit(X_ep, y_ep)
            cat_models[ep] = final_m2
        if HAS_BRF:
            final_m3 = BalancedRandomForestClassifier(
                n_estimators=200, max_features='sqrt',
                sampling_strategy='auto', replacement=False,
                random_state=42, n_jobs=-1,
            )
            final_m3.fit(X_ep, y_ep)
            brf_models[ep] = final_m3
        ens_weights = (0.35, 0.15, 0.50)  

    elif is_rf_strong:
        if HAS_LGB:
            final_m2 = LGBMClassifier(
                n_estimators=300, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8,
                scale_pos_weight=spw,
                random_state=42, n_jobs=-1, verbose=-1,
            )
            final_m2.fit(X_ep, y_ep)
            lgb_models[ep] = final_m2
        final_m3 = RandomForestClassifier(
            n_estimators=200, max_features='sqrt',
            class_weight='balanced', random_state=42, n_jobs=-1,
        )
        final_m3.fit(X_ep, y_ep)
        rf_models[ep]   = final_m3
        ens_weights = (0.45, 0.10, 0.45)  

    else:
        if HAS_LGB:
            final_m2 = LGBMClassifier(
                n_estimators=300, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8,
                scale_pos_weight=spw,
                random_state=42, n_jobs=-1, verbose=-1,
            )
            final_m2.fit(X_ep, y_ep)
            lgb_models[ep] = final_m2
        final_m3 = RandomForestClassifier(
            n_estimators=200, max_features='sqrt',
            class_weight='balanced', random_state=42, n_jobs=-1,
        )
        final_m3.fit(X_ep, y_ep)
        rf_models[ep]   = final_m3
        ens_weights = (0.65, 0.10, 0.25)  

    
    ensemble_models[ep] = {
        'xgb'        : final_xgb2,
        'model2'     : final_m2,
        'model3'     : final_m3,
        'weights'    : ens_weights,
        'is_special' : is_special,
        'tier'       : tier,
    }

    ensemble_results.append({
        'endpoint'    : ep,
        'tier'        : tier,
        'n_rows'      : len(y_ep),
        'n_toxic'     : int((y_ep==1).sum()),
        'xgb_auc'     : round(auc_xgb, 4),
        'm2_auc'      : round(auc_m2,  4),
        'm3_auc'      : round(auc_m3,  4),
        'ensemble_auc': round(auc_ens, 4),
        'delta_vs_xgb': round(auc_ens - auc_xgb, 4),
    })

    print(f'{ep:<22} {tier:<10} {auc_xgb:>7.4f} {auc_m2:>8.4f} '
          f'{auc_m3:>8.4f} {auc_ens:>10.4f}')

ens_df    = pd.DataFrame(ensemble_results)
mean_xgb2 = ens_df['xgb_auc'].mean()
mean_ens  = ens_df['ensemble_auc'].mean()

print('-' * 72)
print(f'{"MEAN":<22} {"":10} {mean_xgb2:>7.4f} {"":>8} {"":>8} {mean_ens:>10.4f}')
print()
print(f'XGB alone   : {mean_xgb2:.4f}')
print(f'Ensemble    : {mean_ens:.4f}')
print(f'Improvement : {mean_ens - mean_xgb2:+.4f}')
print()
for t in ['CAT+BRF', 'RF-heavy', 'XGB-heavy']:
    sub = ens_df[ens_df['tier'] == t]
    if len(sub) > 0:
        print(f'  {t:<12} : mean AUC = {sub["ensemble_auc"].mean():.4f}  '
              f'(delta {(sub["ensemble_auc"]-sub["xgb_auc"]).mean():+.4f} vs XGB)')



## Stage 4 — Per-Endpoint Decision Threshold Optimisation


In [ ]:

from sklearn.metrics import roc_curve, recall_score, precision_score, f1_score

optimal_thresholds = {}
threshold_results  = []

print(f'{"Endpoint":<22} {"Youden thr":>10} {"Recall@0.5":>10} '
      f'{"Recall@J":>9} {"Gain":>7}')
print('-' * 65)

for ep in oof_probs:
    y_true = oof_labels[ep]
    y_prob = oof_probs[ep]

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j_scores  = tpr - fpr
    best_idx  = int(np.argmax(j_scores))
    best_thr  = float(thresholds[best_idx])
    optimal_thresholds[ep] = best_thr

    recall_05  = recall_score(y_true, (y_prob >= 0.5).astype(int),       zero_division=0)
    recall_opt = recall_score(y_true, (y_prob >= best_thr).astype(int),  zero_division=0)
    gain = recall_opt - recall_05

    threshold_results.append({
        'endpoint'   : ep,
        'youden_thr' : round(best_thr, 3),
        'recall_050' : round(recall_05,  3),
        'recall_opt' : round(recall_opt, 3),
        'recall_gain': round(gain, 3),
    })
    marker = '  ← significant gain' if gain > 0.10 else ''
    print(f'{ep:<22} {best_thr:>10.3f} {recall_05:>10.3f} '
          f'{recall_opt:>9.3f} {gain:>+7.3f}{marker}')

thr_df = pd.DataFrame(threshold_results)
thr_df.to_csv('results/optimal_thresholds.csv', index=False)
print(f'\nMean recall gain from threshold optimisation : '
      f'{thr_df["recall_gain"].mean():+.3f}')

with open('models/optimal_thresholds.json', 'w') as f:
    json.dump(optimal_thresholds, f, indent=2)
print('Saved → results/optimal_thresholds.csv')
print('Saved → models/optimal_thresholds.json')


## Result Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax  = axes[0]
x   = np.arange(len(ens_df))
w   = 0.35

bars_ens = ax.bar(x - w/2, ens_df['ensemble_auc'], w,
                  color='#639922', alpha=0.85, label='Ensemble')
bars_xgb = ax.bar(x + w/2, ens_df['xgb_auc'],     w,
                  color='#EF9F27', alpha=0.85, label='XGBoost alone')

for bar, val in zip(bars_ens, ens_df['ensemble_auc']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.3f}', ha='center', fontsize=7, rotation=90)
for bar, val in zip(bars_xgb, ens_df['xgb_auc']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.3f}', ha='center', fontsize=7, rotation=90)

ax.axhline(mean_ens,  color='#3B6D11', linestyle='--', lw=1.5,
           label=f'Ensemble mean {mean_ens:.4f}')
ax.axhline(mean_xgb2, color='#854F0B', linestyle=':',  lw=1.5,
           label=f'XGB mean {mean_xgb2:.4f}')
ax.set_xticks(x)
ax.set_xticklabels(ens_df['endpoint'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.65, 1.02)
ax.set_title('Ensemble vs XGBoost alone', fontsize=11)
ax.legend(fontsize=8)
ax.yaxis.grid(True, alpha=0.3)

ax2 = axes[1]
colors = ['#3B6D11' if d > 0 else '#A32D2D' for d in ens_df['delta_vs_xgb']]
bars_d = ax2.bar(x, ens_df['delta_vs_xgb'], color=colors, alpha=0.85)
for bar, val in zip(bars_d, ens_df['delta_vs_xgb']):
    ypos = bar.get_height()+0.001 if val >= 0 else bar.get_height()-0.004
    ax2.text(bar.get_x()+bar.get_width()/2, ypos,
             f'{val:+.4f}', ha='center', fontsize=8)
ax2.axhline(0, color='black', lw=1, alpha=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(ens_df['endpoint'], rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('AUC gain (Ensemble − XGB alone)')
ax2.set_title('AUC improvement per endpoint', fontsize=11)
ax2.yaxis.grid(True, alpha=0.3)

plt.suptitle(f'Ensemble Results: {mean_xgb2:.4f} → {mean_ens:.4f} '
             f'({mean_ens-mean_xgb2:+.4f})', fontsize=12)
plt.tight_layout()
plt.savefig('plots/ensemble_results.png', dpi=150)
plt.show()
print('Saved → plots/ensemble_results.png')

# ROC curves
fig2, ax3 = plt.subplots(figsize=(9, 7))
for i, ep in enumerate(oof_probs):
    fpr, tpr, _ = roc_curve(oof_labels[ep], oof_probs[ep])
    row     = ens_df[ens_df['endpoint']==ep]
    ens_auc = float(row['ensemble_auc'].values[0]) if len(row)>0 else 0
    ax3.plot(fpr, tpr, color=COLORS[i % len(COLORS)], lw=1.5,
             label=f'{ep}  ({ens_auc:.3f})')
ax3.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate')
ax3.set_title(f'ROC Curves — Ensemble\nMean AUC = {mean_ens:.4f}')
ax3.legend(loc='lower right', fontsize=8, ncol=2)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curves_ensemble.png', dpi=150)
plt.show()
print('Saved → plots/roc_curves_ensemble.png')


---
## Stage 5 — Feature Importance Analysis


In [ ]:

importance_records = []

print('Computing feature importances per endpoint...')
for ep in toxicity_labels:
    if ep not in xgb_models:
        continue
    model    = xgb_models[ep]
    imp_dict = model.get_booster().get_score(importance_type='gain')

    
    for fname, score in imp_dict.items():
        try:
            idx = int(fname[1:])   # strip 'f' prefix
            col = feat_cols[idx] if idx < len(feat_cols) else fname
        except (ValueError, IndexError):
            col = fname
        importance_records.append({
            'endpoint': ep,
            'feature' : col,
            'gain'    : score,
        })

imp_df = pd.DataFrame(importance_records)

# ── Global top-20 features across all endpoints ──────────────────
global_imp = (imp_df.groupby('feature')['gain']
              .mean()
              .sort_values(ascending=False)
              .head(20))

print(f'\nTop 20 features by mean gain across all endpoints:')
print(f'{"Feature":<35} {"Mean Gain":>10}')
print('-' * 48)
for feat, gain in global_imp.items():
    # Classify feature type
    if feat.startswith('morgan'):
        ftype = '[ECFP fingerprint]'
    elif feat.startswith('ap_'):
        ftype = '[Atom Pair fp]'
    elif feat.startswith('tt_'):
        ftype = '[Topo Torsion fp]'
    elif feat.startswith('maccs'):
        ftype = '[MACCS key]'
    elif feat.startswith('smarts'):
        ftype = '[SMARTS fragment]'
    elif feat.startswith('oof_'):
        ftype = '[cross-endpoint]'
    elif feat.startswith('autocorr') or feat.startswith('whim'):
        ftype = '[3D descriptor]'
    elif feat in ['MW','logP','QED','TPSA','HBD','HBA','RotBonds',
                  'ArRings','FracCSP3','HeavyAtoms']:
        ftype = '[physicochemical]'
    else:
        ftype = '[descriptor]'
    print(f'{feat:<35} {gain:>10.2f}  {ftype}')

# ── Per-endpoint top-5 features ──────────────────────────────────
print(f'\n{"="*65}')
print('Top 5 features per endpoint:')
print(f'{"="*65}')
for ep in toxicity_labels:
    ep_imp = (imp_df[imp_df['endpoint']==ep]
              .sort_values('gain', ascending=False)
              .head(5))
    if len(ep_imp) == 0:
        continue
    print(f'\n{ep}:')
    for _, row in ep_imp.iterrows():
        print(f'  {row["feature"]:<35} gain={row["gain"]:.2f}')

# ── Feature category breakdown ────────────────────────────────────
def categorise(feat):
    if feat.startswith('morgan'):   return 'Morgan fingerprint'
    if feat.startswith('ap_'):      return 'Atom Pair fp'
    if feat.startswith('tt_'):      return 'Topo Torsion fp'
    if feat.startswith('maccs'):    return 'MACCS keys'
    if feat.startswith('smarts'):   return 'SMARTS fragments'
    if feat.startswith('oof_'):     return 'Cross-endpoint OOF'
    if feat.startswith('autocorr'): return '3D AUTOCORR'
    if feat.startswith('whim'):     return '3D WHIM'
    if feat in ['MW','logP','QED','TPSA','HBD','HBA',
                'RotBonds','ArRings','FracCSP3','HeavyAtoms']:
        return 'Core physicochemical'
    if feat.startswith('lip_') or feat in ['logP_x_MW','TPSA_per_MW',
                                            'HBD_HBA_sum','logMW']:
        return 'Engineered features'
    if feat.startswith('chiral') or feat.startswith('total_charge'):
        return 'Chirality/charge'
    return 'Extended descriptors'

imp_df['category'] = imp_df['feature'].apply(categorise)
cat_summary = (imp_df.groupby('category')['gain']
               .agg(['mean','count'])
               .sort_values('mean', ascending=False))
cat_summary.columns = ['Mean Gain', 'N Features Used']

print(f'\n{"="*55}')
print('Feature category contribution summary:')
print(f'{"="*55}')
print(cat_summary.round(2).to_string())

imp_df.to_csv('results/feature_importances.csv', index=False)
cat_summary.to_csv('results/feature_category_summary.csv')
print('\nSaved → results/feature_importances.csv')
print('Saved → results/feature_category_summary.csv')

# ── Importance bar chart ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Global top-20
ax = axes[0]
feat_names = [f[:28]+'…' if len(f)>28 else f for f in global_imp.index]
bars = ax.barh(range(len(global_imp)), global_imp.values,
               color=COLORS[:len(global_imp)] if len(global_imp)<=len(COLORS)
               else ['#378ADD']*len(global_imp), alpha=0.85)
ax.set_yticks(range(len(global_imp)))
ax.set_yticklabels(feat_names, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Mean XGBoost Gain')
ax.set_title('Top 20 Features Linked to Toxicity\n(mean gain across all endpoints)', fontsize=10)
ax.xaxis.grid(True, alpha=0.3)

# Category pie
ax2 = axes[1]
cat_vals = cat_summary['Mean Gain'].values
cat_labs = [f'{idx}\n({v:.0f})' for idx, v in zip(cat_summary.index, cat_vals)]
ax2.pie(cat_vals, labels=cat_labs, colors=COLORS[:len(cat_vals)],
        autopct='%1.1f%%', startangle=140, textprops={'fontsize': 8})
ax2.set_title('Feature Category Contribution\n(by mean XGBoost gain)', fontsize=10)

plt.suptitle('Structural Features Linked to Toxicity — Feature Importance Analysis',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/feature_importance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/feature_importance_analysis.png')


---
## Stage 5 — Prediction Interface for New Compounds

A standalone function that accepts a SMILES string and returns toxicity probability
and binary prediction for all 12 Tox21 endpoints, using the optimal per-endpoint
decision threshold computed in Stage 3.

This addresses the hackathon requirement: *"build a simple prediction interface
or visualisation for evaluating new compounds"*.

In [ ]:
# ════════════════════════════════════════════════════════════════
# STAGE 5 — Prediction Interface
# Accepts any SMILES string → returns per-endpoint toxicity probs
# and binary predictions using the optimal Youden J thresholds.
# ════════════════════════════════════════════════════════════════

def build_features_for_smiles(smiles_str):
    """Compute the full feature vector for a single SMILES string.
    Returns a (1, n_features) numpy array matching feat_cols order,
    or None if the SMILES is invalid."""
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return None

    # ── Fingerprints ─────────────────────────────────────────────
    morgan_gen_ = rdFingerprintGenerator.GetMorganGenerator(
        radius=2, fpSize=2048, includeChirality=True)
    morgan_r2 = list(morgan_gen_.GetFingerprint(mol))

    ap_fp = list(rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(
        mol, nBits=1024, includeChirality=True))
    tt_fp = list(rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(
        mol, nBits=1024, includeChirality=True))
    maccs_fp = list(MACCSkeys.GenMACCSKeys(mol))

    fp_raw = morgan_r2 + ap_fp + tt_fp + maccs_fp
    # Apply same VarianceThreshold fitted on training data
    fp_kept = np.array(fp_raw)[vt.get_support()].tolist()

    # ── Continuous descriptors ────────────────────────────────────
    desc_row  = compute_descriptors(mol)
    ext_row   = compute_extended_descriptors(mol)
    smarts_row= compute_smarts_features(mol)
    chiral_row= compute_chirality_charge(mol)
    zinc_row  = compute_zinc_features(mol)

    # Engineered features
    mw_   = desc_row.get('MW', np.nan)
    logp_ = desc_row.get('logP', np.nan)
    tpsa_ = desc_row.get('TPSA', np.nan)
    hbd_  = desc_row.get('HBD', np.nan)
    hba_  = desc_row.get('HBA', np.nan)
    eng_row = {
        'lip_pass'      : int(mw_<500 and logp_<5 and hbd_<=5 and hba_<=10),
        'lip_violations': int(mw_>=500)+int(logp_>=5)+int(hbd_>5)+int(hba_>10),
        'logP_x_MW'     : logp_ * mw_,
        'TPSA_per_MW'   : tpsa_ / (mw_ + 1),
        'HBD_HBA_sum'   : hbd_ + hba_,
        'logMW'         : np.log1p(mw_),
    }

    # Combine all continuous columns in training order
    cont_row = {}
    for col in continuous_df.columns:
        if col in desc_row:    cont_row[col] = desc_row[col]
        elif col in ext_row:   cont_row[col] = ext_row[col]
        elif col in smarts_row:cont_row[col] = smarts_row[col]
        elif col in eng_row:   cont_row[col] = eng_row[col]
        elif col in zinc_row:  cont_row[col] = zinc_row[col]
        elif col in chiral_row:cont_row[col] = chiral_row[col]
        else:                  cont_row[col] = np.nan

    cont_vals = np.array([cont_row.get(c, np.nan) for c in continuous_df.columns],
                         dtype=np.float32)
    cont_vals = np.nan_to_num(cont_vals, nan=0.0, posinf=0.0, neginf=0.0)

    row = np.concatenate([np.array(fp_kept, dtype=np.float32), cont_vals])
    return row.reshape(1, -1)


def predict_toxicity(smiles_str, verbose=True):
    """
    Predict Tox21 toxicity for a SMILES string.

    Returns a DataFrame with columns:
        endpoint, probability, prediction, threshold, risk_level
    """
    X_new = build_features_for_smiles(smiles_str)
    if X_new is None:
        print(f'Invalid SMILES: {smiles_str}')
        return None

    results = []
    for ep in toxicity_labels:
        if ep not in xgb_models:
            continue
        model = ensemble_models.get(ep)
        if model is None:
            prob = xgb_models[ep].predict_proba(X_new)[0, 1]
        else:
            w   = model['weights']
            p_x = model['xgb'].predict_proba(X_new)[0, 1]
            p_m2 = model['model2'].predict_proba(X_new)[0, 1] if model['model2'] else p_x
            p_m3 = model['model3'].predict_proba(X_new)[0, 1] if model['model3'] else p_x
            prob = w[0]*p_x + w[1]*p_m2 + w[2]*p_m3

        thr  = optimal_thresholds.get(ep, 0.5)
        pred = int(prob >= thr)
        risk = 'HIGH' if prob > 0.7 else ('MODERATE' if prob > 0.4 else 'LOW')

        results.append({
            'endpoint'   : ep,
            'probability': round(float(prob), 4),
            'prediction' : 'TOXIC' if pred == 1 else 'SAFE',
            'threshold'  : round(thr, 3),
            'risk_level' : risk,
        })

    result_df = pd.DataFrame(results)

    if verbose:
        print(f'\nToxicity Prediction for: {smiles_str}')
        print('=' * 65)
        print(f'{"Endpoint":<22} {"Prob":>6} {"Pred":>7} {"Risk":>10}  {"Thr":>6}')
        print('-' * 55)
        for _, row in result_df.iterrows():
            marker = ' ◄' if row['prediction'] == 'TOXIC' else ''
            print(f'{row["endpoint"]:<22} {row["probability"]:>6.4f} '
                  f'{row["prediction"]:>7} {row["risk_level"]:>10}  '
                  f'{row["threshold"]:>6.3f}{marker}')
        n_toxic = (result_df['prediction'] == 'TOXIC').sum()
        print(f'\nSummary: {n_toxic}/{len(result_df)} endpoints predicted TOXIC')
        mean_prob = result_df['probability'].mean()
        print(f'Mean toxicity probability across all endpoints: {mean_prob:.4f}')
    return result_df


# ── Visualisation helper ──────────────────────────────────────────
def visualise_prediction(smiles_str):
    """Bar chart of predicted toxicity probabilities across all endpoints."""
    result_df = predict_toxicity(smiles_str, verbose=False)
    if result_df is None:
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    colors_ = ['#A32D2D' if p == 'TOXIC' else '#3B6D11'
               for p in result_df['prediction']]
    bars = ax.bar(result_df['endpoint'], result_df['probability'],
                  color=colors_, alpha=0.85, edgecolor='white', linewidth=0.5)
    # Threshold markers
    for i, (_, row) in enumerate(result_df.iterrows()):
        ax.plot([i-0.4, i+0.4], [row['threshold'], row['threshold']],
                'k--', lw=1.2, alpha=0.6)
    ax.set_xticks(range(len(result_df)))
    ax.set_xticklabels(result_df['endpoint'], rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Predicted Toxicity Probability')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'Toxicity Profile: {smiles_str[:60]}', fontsize=10)
    ax.yaxis.grid(True, alpha=0.3)

    from matplotlib.patches import Patch
    legend_elems = [Patch(facecolor='#A32D2D', label='Predicted TOXIC'),
                    Patch(facecolor='#3B6D11', label='Predicted SAFE'),
                    plt.Line2D([0],[0], color='black', linestyle='--',
                               label='Endpoint threshold')]
    ax.legend(handles=legend_elems, fontsize=9)
    plt.tight_layout()
    plt.savefig('plots/prediction_profile.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → plots/prediction_profile.png')


# ── Demo: run on 3 example compounds ─────────────────────────────
print('PREDICTION INTERFACE — DEMO')
print('=' * 65)

DEMO_COMPOUNDS = {
    'Bisphenol A (known endocrine disruptor)' :
        'CC(C)(c1ccc(O)cc1)c1ccc(O)cc1',
    'Caffeine (generally safe)' :
        'Cn1cnc2c1c(=O)n(C)c(=O)n2C',
    'Tamoxifen (ER antagonist)' :
        'CCC(=C(c1ccccc1)c1ccc(OCCN(C)C)cc1)c1ccccc1',
}

for name, smi in DEMO_COMPOUNDS.items():
    print(f'\n--- {name} ---')
    res = predict_toxicity(smi, verbose=True)

print('\n--- Visualising Bisphenol A ---')
visualise_prediction('CC(C)(c1ccc(O)cc1)c1ccc(O)cc1')
print('\nTo evaluate your own compound:')
print('  result = predict_toxicity("YOUR_SMILES_HERE")')
print('  visualise_prediction("YOUR_SMILES_HERE")')


---
## Results — ROC-AUC Comparison Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax  = axes[0]
x   = np.arange(len(ens_df))
w   = 0.35

bars_ens = ax.bar(x - w/2, ens_df['ensemble_auc'], w,
                  color='#639922', alpha=0.85, label='Ensemble')
bars_xgb = ax.bar(x + w/2, ens_df['xgb_auc'],     w,
                  color='#EF9F27', alpha=0.85, label='XGBoost alone')

for bar, val in zip(bars_ens, ens_df['ensemble_auc']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.3f}', ha='center', fontsize=7, rotation=90)
for bar, val in zip(bars_xgb, ens_df['xgb_auc']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.3f}', ha='center', fontsize=7, rotation=90)

ax.axhline(mean_ens,  color='#3B6D11', linestyle='--', lw=1.5,
           label=f'Ensemble mean {mean_ens:.4f}')
ax.axhline(mean_xgb2, color='#854F0B', linestyle=':',  lw=1.5,
           label=f'XGB mean {mean_xgb2:.4f}')
ax.set_xticks(x)
ax.set_xticklabels(ens_df['endpoint'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.65, 1.02)
ax.set_title('Ensemble vs XGBoost alone', fontsize=11)
ax.legend(fontsize=8)
ax.yaxis.grid(True, alpha=0.3)

ax2 = axes[1]
colors = ['#3B6D11' if d > 0 else '#A32D2D' for d in ens_df['delta_vs_xgb']]
bars_d = ax2.bar(x, ens_df['delta_vs_xgb'], color=colors, alpha=0.85)
for bar, val in zip(bars_d, ens_df['delta_vs_xgb']):
    ypos = bar.get_height()+0.001 if val >= 0 else bar.get_height()-0.004
    ax2.text(bar.get_x()+bar.get_width()/2, ypos,
             f'{val:+.4f}', ha='center', fontsize=8)
ax2.axhline(0, color='black', lw=1, alpha=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(ens_df['endpoint'], rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('AUC gain (Ensemble − XGB alone)')
ax2.set_title('AUC improvement per endpoint', fontsize=11)
ax2.yaxis.grid(True, alpha=0.3)

plt.suptitle(f'Ensemble Results: {mean_xgb2:.4f} → {mean_ens:.4f} '
             f'({mean_ens-mean_xgb2:+.4f})', fontsize=12)
plt.tight_layout()
plt.savefig('plots/ensemble_results.png', dpi=150)
plt.show()
print('Saved → plots/ensemble_results.png')

# ROC curves
fig2, ax3 = plt.subplots(figsize=(9, 7))
for i, ep in enumerate(oof_probs):
    fpr, tpr, _ = roc_curve(oof_labels[ep], oof_probs[ep])
    row     = ens_df[ens_df['endpoint']==ep]
    ens_auc = float(row['ensemble_auc'].values[0]) if len(row)>0 else 0
    ax3.plot(fpr, tpr, color=COLORS[i % len(COLORS)], lw=1.5,
             label=f'{ep}  ({ens_auc:.3f})')
ax3.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate')
ax3.set_title(f'ROC Curves — Ensemble\nMean AUC = {mean_ens:.4f}')
ax3.legend(loc='lower right', fontsize=8, ncol=2)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curves_ensemble.png', dpi=150)
plt.show()
print('Saved → plots/roc_curves_ensemble.png')


---
## Save All Outputs

In [ ]:
with open('models/ensemble_models.pkl', 'wb') as f:
    pickle.dump(ensemble_models, f)
with open('models/xgb_models.pkl', 'wb') as f:
    pickle.dump(xgb_models, f)
if cat_models:
    with open('models/cat_models.pkl', 'wb') as f:
        pickle.dump(cat_models, f)
if brf_models:
    with open('models/brf_models.pkl', 'wb') as f:
        pickle.dump(brf_models, f)
with open('models/spw_dict.json', 'w') as f:
    json.dump(spw_dict, f, indent=2)

ens_df.to_csv('results/ensemble_results.csv', index=False)
thr_df.to_csv('results/optimal_thresholds.csv', index=False)
with open('models/optimal_thresholds.json', 'w') as f:
    json.dump(optimal_thresholds, f, indent=2)

best_ep  = ens_df.loc[ens_df['ensemble_auc'].idxmax()]
worst_ep = ens_df.loc[ens_df['ensemble_auc'].idxmin()]
most_imp = ens_df.loc[ens_df['delta_vs_xgb'].idxmax()]

print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'  Fingerprint bits (after VT) : {fp_filtered_df.shape[1]:,}')
print(f'  Continuous features         : {cont_imputed.shape[1]:,}')
print(f'  TOTAL features              : {len(feat_cols):,}')
print()
print(f'  XGB alone mean AUC          : {mean_xgb2:.4f}')
print(f'  Ensemble mean AUC           : {mean_ens:.4f}')
print(f'  Improvement vs XGB          : {mean_ens - mean_xgb2:+.4f}')
print()
print(f'  Best endpoint  : {best_ep["endpoint"]}  (AUC={best_ep["ensemble_auc"]:.4f})')
print(f'  Worst endpoint : {worst_ep["endpoint"]}  (AUC={worst_ep["ensemble_auc"]:.4f})')
print(f'  Most improved  : {most_imp["endpoint"]}  (+{most_imp["delta_vs_xgb"]:.4f} vs XGB)')
print()
print('Saved:')
print('  models/ensemble_models.pkl')
print('  models/xgb_models.pkl')
print('  results/ensemble_results.csv')
print('  plots/ensemble_results.png')
print('  plots/roc_curves_ensemble.png')
print('  features_raw.csv')
print('  results/optimal_thresholds.csv')
print('  results/feature_importances.csv')
print('  plots/feature_importance_analysis.png')
print('  plots/prediction_profile.png')
print()
print('To predict a new molecule:')
print("  with open('models/ensemble_models.pkl','rb') as f:")
print("      models = pickle.load(f)")
print("  ep_models = models['NR-AR']")
print("  w = ep_models['weights']")
print("  p_xgb = ep_models['xgb'].predict_proba(X_new)[:,1]")
print("  p_lgb = ep_models['lgb'].predict_proba(X_new)[:,1] if ep_models['lgb'] else p_xgb")
print("  p_rf  = ep_models['rf'].predict_proba(X_new)[:,1]")
print("  prob  = w[0]*p_xgb + w[1]*p_lgb + w[2]*p_rf")
